# LambdaG Visualization — X2_ecclesiastes vs. Known Authors

This notebook runs the token-level LambdaG visualization on the hold-out work "X2_ecclesiastes" against a balanced sample of known works from each author, with a reference set of sentences from other authors.

## Approach

We will create visualizations for each author, comparing X2_ecclesiastes to that author's known works.
Balanced sampling uses `TwoLevelCorpus.sample_balanced()` which samples sentences evenly across works for the known author, and evenly across authors → works for the reference set.

Note: Since we don't have a ThreeLevelCorpus with translator level, we adapt the approach to work with TwoLevelCorpus (author, work).

## Stage 1: Setup

Load required libraries and define paths.

In [3]:
import numpy as np
from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# Load corpora
train_corpus = TwoLevelCorpus.from_pickle("train_corpus.pkl")
test_corpus = TwoLevelCorpus.from_pickle("test_corpus.pkl")
holdout_corpus = TwoLevelCorpus.from_pickle("ecclesiastes_corpus.pkl")  # This is the work we held out

print(f"train_corpus: {len(train_corpus)} sentences")
print(f"test_corpus:  {len(test_corpus)} sentences")
print(f"holdout_corpus (X2_ecclesiastes): {len(holdout_corpus)} sentences")
print(f"\nAuthors in test_corpus: {sorted(test_corpus.get_unique_values('author'))}")
print(f"\nHold-out work author: {sorted(set(s.metadata['author'] for s in holdout_corpus))}")

train_corpus: 33085 sentences
test_corpus:  14141 sentences
holdout_corpus (X2_ecclesiastes): 79 sentences

Authors in test_corpus: ['AI', 'AeC', 'AeP', 'B', 'BC', 'DC', 'E', 'FB', 'GAp', 'GAr', 'GC', 'GM', 'GP', 'GVS', 'GiP', 'H', 'HA', 'HP', 'IH', 'II', 'M', 'NLC', 'OM', 'OSE', 'RC', 'RT', 'SG', 'U1', 'U10', 'U11', 'U2', 'U3', 'U4', 'U5', 'U6', 'U7', 'U8', 'U9', 'X3', 'X4']

Hold-out work author: ['X2']


In [4]:
# Get authors that have works in the test set
authors_with_works = test_corpus.get_unique_values("author")
print(f"Found {len(authors_with_works)} authors with works in test set")

# Set up random number generator for reproducibility
rng = np.random.default_rng(seed=42)

# Process each author
for author in sorted(authors_with_works):
    print(f"\n{'='*60}")
    print(f"Processing author: {author}")
    print(f"{'='*60}")

    # --- Balanced sampling using TwoLevelCorpus.sample_balanced ---
    # Known author: current author from test_corpus
    # Sample ~2000 sentences balanced across his/her works
    known_corpus = test_corpus.filter_by_level("author", author)
    if len(known_corpus) == 0:
        print(f"  Skipping {author}: no works in test set")
        continue
    print(f"  {author}'s works in test_corpus: {known_corpus.get_unique_values('work')}")
    for work in known_corpus.get_unique_values("work"):
        work_corpus = known_corpus.filter_by_level("work", work)
        print(f"    {work}: {len(work_corpus)} sentences")

    known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
    sentences=2000,
    rng=rng,
    )
    known_sentences = [list(s.text.split()) for s in known_sentences_raw]
    print(f"\n  Sampled {len(known_sentences)} known sentences")
    print(f"  Breakdown: {known_breakdown}")

    # Reference corpus: 10000 sentences from all OTHER authors in test_corpus
    ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])

    ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(
    sentences=10000,
    rng=rng,
    )

    ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]
    print(f"  Sampled {len(ref_sentences)} reference sentences")
    print(f"  Reference breakdown: {ref_breakdown}")

    # --- Prepare questioned sentences from holdout_corpus (X_ordine) ---
    # q_sentences: tokenized sentences for the visualization
    q_sentences = [s.text.split() for s in holdout_corpus]
    print(f"  Questioned sentences (X_ordine): {len(q_sentences)}")

    # Try to load unmasked sentences from quesplit.txt for colour-coding
    # We need to construct the filename from the author and work
    # Since X_ordine is the hold-out work, we look for its quesplit file
    unmasked_sents = None
    # Get the work from the holdout corpus (should be X_ordine for all sentences)
    holdout_work = holdout_corpus[0].metadata["work"] if len(holdout_corpus) > 0 else "X_ordine"
    # Construct expected filename pattern
    import os
    data_dir = "data_processed"
    # Look for files matching *_X_ordine_sentences.quesplit.txt
    quesplit_pattern = f"*{holdout_work}_sentences.quesplit.txt"
    import glob
    quesplit_files = glob.glob(os.path.join(data_dir, quesplit_pattern))
    if quesplit_files:
        quesplit_file = quesplit_files[0]
        try:
            with open(quesplit_file, "r", encoding="utf-8") as f:
                unmasked_sents = [line.strip().split() for line in f if line.strip()]
            print(f"  Unmasked sentences loaded from {os.path.basename(quesplit_file)}: {len(unmasked_sents)}")
            # Verify lengths match
            if len(q_sentences) != len(unmasked_sents):
                print(f"  WARNING: Mismatch! q_sentences: {len(q_sentences)}, unmasked_sents: {len(unmasked_sents)}")
                unmasked_sents = None  # Fall back to masked only
        except Exception as e:
            print(f"  Could not load unmasked sentences: {e}")
            unmasked_sents = None
    else:
        print(f"  No quesplit file found for pattern: {quesplit_pattern}")

    # --- Run LambdaG visualization (masked tokens) ---
    print(f"\n  Running LambdaG visualization (masked)...")
    result = lambdaG_visualize(
        q_text=q_sentences,
        k_text=known_sentences,
        ref_text=ref_sentences,
        N=8,
        r=30,
        output="html",
        scale="relative",
        negative=True,
        order_by="importance",
        rng=rng,
        print_path=f"html_ecclesiastes/lambdag_output_{author}_masked.html",
    )
    print(f"  HTML output saved to: lambdag_output_{author}_masked.html")
    print(f"  Table shape: {result['table'].shape}")
    print(f"  Sentence count: {result['table']['sentence_id'].nunique()}")

    # --- Run LambdaG visualization (unmasked words) if available ---
    if unmasked_sents is not None:
        print(f"\n  Running LambdaG visualization (unmasked)...")
        result_unmasked = lambdaG_visualize(
            q_text=q_sentences,
            k_text=known_sentences,
            ref_text=ref_sentences,
            N=8,
            r=30,
            output="html",
            scale="relative",
            negative=True,
            order_by="",
            unmasked_sentences=unmasked_sents,
            rng=rng,
            print_path=f"html_ecclesiastes/lambdag_output_{author}_unmasked.html",
        )
        print(f"  HTML output saved to: lambdag_output_{author}_unmasked.html")
    else:
        print(f"\n  Skipping unmasked visualization (no unmasked sentences available)")

    # --- Display summary statistics ---
    print(f"\n  Summary Statistics (masked):")
    table = result["table"]
    n_sentences = table["sentence_id"].nunique()
    n_tokens = len(table)
    print(f"    Number of sentences: {n_sentences}")
    print(f"    Number of tokens: {n_tokens}")
    print(f"    Mean sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].mean():.3f}")
    print(f"    Std sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].std():.3f}")
    print(f"    Positive tokens: {(table['lambdaG'] > 0).sum()} ({(table['lambdaG'] > 0).mean()*100:.1f}%)")
    print(f"    Negative tokens: {(table['lambdaG'] < 0).sum()} ({(table['lambdaG'] < 0).mean()*100:.1f}%)")

    if unmasked_sents is not None:
        print(f"\n  Summary Statistics (unmasked):")
        table = result_unmasked["table"]
        n_sentences = table["sentence_id"].nunique()
        n_tokens = len(table)
        print(f"    Number of sentences: {n_sentences}")
        print(f"    Number of tokens: {n_tokens}")
        print(f"    Mean sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].mean():.3f}")
        print(f"    Std sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].std():.3f}")
        print(f"    Positive tokens: {(table['lambdaG'] > 0).sum()} ({(table['lambdaG'] > 0).mean()*100:.1f}%)")
        print(f"    Negative tokens: {(table['lambdaG'] < 0).sum()} ({(table['lambdaG'] < 0).mean()*100:.1f}%)")

    print(f"\n  Completed visualizations for {author}")

Found 40 authors with works in test set

Processing author: AI
  AI's works in test_corpus: ['anticlaudianus']
    anticlaudianus: 470 sentences

  Sampled 470 known sentences
  Breakdown: {'AI': {'anticlaudianus': 9861}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AeC': {'medicaminum': 9791, 'viaticus': 4954}, 'AeP': {'karolinus': 5806}, 'B': {'judas': 1991}, 'BC': {'contemptu': 6994}, 'DC': {'mathildis': 6449}, 'E': {'biblia': 3326}, 'FB': {'nuptis': 10684}, 'GAp': {'roberti': 6004}, 'GAr': {'philippis': 11367}, 'GC': {'alexandreis': 12123}, 'GM': {'merlini': 3379}, 'GP': {'ligurinus': 14417}, 'GVS': {'poetria': 4958}, 'GiP': {'historia': 5741}, 'H': {'epigrammata': 907, 'maria': 1922}, 'HA': {'planctus': 5049}, 'HP': {'maiolichinus': 7640}, 'IH': {'architrenius': 9514}, 'II': {'ylias': 8192}, 'M': {'capitulorum': 2954, 'maurilii': 1415, 'victoris': 1158}, 'NLC': {'passio': 2286}, 'OM': {'ernestus': 3290}, 'OSE': {'doctrina': 1454}, 'RC': {'tancredi': 876}, 'RT': {'m

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_5835/98330203.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_AI_masked.html
  Table shape: (1158, 8)
  Sentence count: 79

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 79
    Number of tokens: 1158
    Mean sentence λG: -0.460
    Std sentence λG: 1.252
    Positive tokens: 579 (50.0%)
    Negative tokens: 579 (50.0%)

  Completed visualizations for AI

Processing author: AeC
  AeC's works in test_corpus: ['medicaminum', 'viaticus']
    medicaminum: 292 sentences
    viaticus: 217 sentences

  Sampled 509 known sentences
  Breakdown: {'AeC': {'medicaminum': 9791, 'viaticus': 4954}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AI': {'anticlaudianus': 9861}, 'AeP': {'karolinus': 5806}, 'B': {'judas': 1991}, 'BC': {'contemptu': 6994}, 'DC': {'mathildis': 6449}, 'E': {'biblia': 3326}, 'FB': {'nuptis': 10684}, 'GAp': {'roberti': 6004}, 'GAr': {'philippis': 11367}, 'GC': {'alexandreis': 12123}, 'GM': {'merlini': 3379},

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_5835/98330203.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_AeC_masked.html
  Table shape: (1158, 8)
  Sentence count: 79

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 79
    Number of tokens: 1158
    Mean sentence λG: -0.385
    Std sentence λG: 1.485
    Positive tokens: 588 (50.8%)
    Negative tokens: 570 (49.2%)

  Completed visualizations for AeC

Processing author: AeP
  AeP's works in test_corpus: ['karolinus']
    karolinus: 190 sentences

  Sampled 190 known sentences
  Breakdown: {'AeP': {'karolinus': 5806}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AI': {'anticlaudianus': 9861}, 'AeC': {'medicaminum': 9791, 'viaticus': 4954}, 'B': {'judas': 1991}, 'BC': {'contemptu': 6994}, 'DC': {'mathildis': 6449}, 'E': {'biblia': 3326}, 'FB': {'nuptis': 10684}, 'GAp': {'roberti': 6004}, 'GAr': {'philippis': 11367}, 'GC': {'alexandreis': 12123}, 'GM': {'merlini': 3379}, 'GP': {'ligurinus': 14417}, 'GVS': {'poet

...

  HTML output saved to: lambdag_output_U9_masked.html
  Table shape: (1158, 8)
  Sentence count: 79

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 79
    Number of tokens: 1158
    Mean sentence λG: -0.114
    Std sentence λG: 1.523
    Positive tokens: 525 (45.3%)
    Negative tokens: 633 (54.7%)

  Completed visualizations for U9

Processing author: X3
  X3's works in test_corpus: ['machabeis']
    machabeis: 61 sentences

  Sampled 61 known sentences
  Breakdown: {'X3': {'machabeis': 1086}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AI': {'anticlaudianus': 9861}, 'AeC': {'medicaminum': 9791, 'viaticus': 4954}, 'AeP': {'karolinus': 5806}, 'B': {'judas': 1991}, 'BC': {'contemptu': 6994}, 'DC': {'mathildis': 6449}, 'E': {'biblia': 3326}, 'FB': {'nuptis': 10684}, 'GAp': {'roberti': 6004}, 'GAr': {'philippis': 11367}, 'GC': {'alexandreis': 12123}, 'GM': {'merlini': 3379}, 'GP': {'ligurinus': 

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_5835/98330203.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_X3_masked.html
  Table shape: (1158, 8)
  Sentence count: 79

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 79
    Number of tokens: 1158
    Mean sentence λG: 0.229
    Std sentence λG: 1.407
    Positive tokens: 578 (49.9%)
    Negative tokens: 580 (50.1%)

  Completed visualizations for X3

Processing author: X4
  X4's works in test_corpus: ['certamina']
    certamina: 14 sentences

  Sampled 14 known sentences
  Breakdown: {'X4': {'certamina': 306}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AI': {'anticlaudianus': 9861}, 'AeC': {'medicaminum': 9791, 'viaticus': 4954}, 'AeP': {'karolinus': 5806}, 'B': {'judas': 1991}, 'BC': {'contemptu': 6994}, 'DC': {'mathildis': 6449}, 'E': {'biblia': 3326}, 'FB': {'nuptis': 10684}, 'GAp': {'roberti': 6004}, 'GAr': {'philippis': 11367}, 'GC': {'alexandreis': 12123}, 'GM': {'merlini': 3379}, 'GP': {'ligurinus': 14

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_5835/98330203.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_X4_masked.html
  Table shape: (1158, 8)
  Sentence count: 79

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 79
    Number of tokens: 1158
    Mean sentence λG: -0.220
    Std sentence λG: 1.003
    Positive tokens: 564 (48.7%)
    Negative tokens: 594 (51.3%)

  Completed visualizations for X4


## Stage 3: Combined Visualization (Optional)

If desired, we can also create a single visualization comparing *Ecclesiastes* to ALL known works combined,
with a reference set from other works. This would be similar to the Nile approach but at the author level.

This is commented out by default but can be uncommented if needed.

In [6]:
# OPTIONAL: Combined visualization - X2_ecclesiastes vs all known works
print("\n" + "=" * 60)
print("RUNNING COMBINED VISUALIZATION: X2_ecclesiastes vs ALL known works")
print("=" * 60)

# Known corpus: all works contained in test_corpus
known_corpus = test_corpus

# Sample 2000 sentences, balanced across the corpus
known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
    sentences=2000,
    rng=rng
)

# Convert sentences to token lists
known_sentences = [
    list(s.text.split())
    for s in known_sentences_raw
]

print(f"\nSampled {len(known_sentences)} known sentences")

# Show corpus composition
print("\nKnown corpus breakdown:")
if isinstance(known_breakdown, dict):
    for key, value in known_breakdown.items():
        print(f"  {key}: {value}")
else:
    print(known_breakdown)

# Reference corpus
# A separate reference set is needed for the combined visualization.
# This could later be train_corpus, holdout_corpus, or another independent corpus.
print("\nReference set creation skipped for combined visualization.")
print("To run combined visualization, implement reference set selection.")


RUNNING COMBINED VISUALIZATION: X2_ecclesiastes vs ALL known works

Sampled 2000 known sentences

Known corpus breakdown:
  AI: {'anticlaudianus': 9861}
  AeC: {'medicaminum': 9791, 'viaticus': 2311}
  AeP: {'karolinus': 843}
  B: {'judas': 288}
  BC: {'contemptu': 974}
  DC: {'mathildis': 715}
  E: {'biblia': 448}
  FB: {'nuptis': 1169}
  GAp: {'roberti': 637}
  GAr: {'philippis': 1301}
  GC: {'alexandreis': 1187}
  GM: {'merlini': 411}
  GP: {'ligurinus': 1435}
  GVS: {'poetria': 472}
  GiP: {'historia': 496}
  H: {'epigrammata': 125, 'maria': 110}
  HA: {'planctus': 439}
  HP: {'maiolichinus': 654}
  IH: {'architrenius': 871}
  II: {'ylias': 800}
  M: {'capitulorum': 392, 'maurilii': 40, 'victoris': 16}
  NLC: {'passio': 424}
  OM: {'ernestus': 545}
  OSE: {'doctrina': 233}
  RC: {'tancredi': 86}
  RT: {'mauri': 102}
  SG: {'ecclesiasten': 34}
  U1: {'canticamariae': 38}
  U10: {'karolellus': 324}
  U11: {'katherinae': 611}
  U2: {'liberesdre': 86}
  U3: {'generatione': 152}
  U4: 